# UVP seminarska naloga – analiza prenosnikov

## Analiza obnovljenih prenosnikov na spletni strani Refurb.si

V seminarski nalogi sem analiziral ponudbo obnovljenih prenosnikov na spletni strani Refurb.si. Analizo sem omejil na prenosnike s procesorjem Intel Core i7. S spletne strani sem s pomočjo knjižnic `requests` in `BeautifulSoup` prenesel HTML-kodo posameznih strani, nato pa iz nje izločil ime naprave, procesor, količino RAM-a, velikost SSD-ja, velikost zaslona, grafično kartico, operacijski sistem, ceno in povezavo do izdelka.

Pridobil sem podatke o 73 različnih prenosnikih in devetih atributih. Največji problem pri podatkih je bil zapis številskih vrednosti. Cena je bila zapisana kot besedilo z znakom za evro, SSD pa je bil pri nekaterih napravah izražen v gigabajtih in pri drugih v terabajtih. Pred analizo sem zato cene pretvoril v števila, SSD-je v gigabajte ter iz zapisov za RAM in zaslon odstranil merske enote.

Pri analizi sem želel ugotoviti, kako so cena, RAM, SSD, velikost zaslona in namenska grafična kartica povezani med seboj. Rezultate sem prikazal s tabelami in grafikoni, izdelanimi s knjižnicama `pandas` in `matplotlib`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

podatki = pd.read_csv('prenosniki.csv')

# Cena: na primer 1.499,00 € pretvorimo v 1499.0.
podatki['cena_eur'] = (
    podatki['cena']
    .str.replace(' €', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

podatki['ram_gb'] = podatki['ram'].str.extract(r'(\d+)', expand=False).astype(float)
podatki['zaslon_in'] = podatki['zaslon'].str.replace('\"', '', regex=False).astype(float)

def pretvori_ssd(vrednost):
    if pd.isna(vrednost):
        return None
    stevilo, enota = vrednost.split()
    stevilo = float(stevilo)
    return stevilo * 1000 if enota == 'TB' else stevilo

podatki['ssd_gb'] = podatki['ssd'].apply(pretvori_ssd)
podatki['namenska_graficna'] = podatki['graficna'].notna()
podatki.head()

## Osnovni pregled podatkov

Najprej sem preveril velikost podatkovne množice, manjkajoče vrednosti in osnovne statistike številskih atributov. V množici je 73 prenosnikov. Vsi imajo zapisano ceno, RAM in velikost zaslona, pri treh manjka podatek o SSD-ju, pri enem pa podatek o operacijskem sistemu. Prazen podatek o grafični kartici sem obravnaval kot napravo brez zaznane namenske grafične kartice.

Povprečna cena prenosnika znaša 1.250,51 €, mediana pa 1.199 €. Najcenejši prenosnik stane 499 €, najdražji pa 2.999 €. Večina ponudbe je tako precej bližje mediani kot najvišji ceni, najdražji prenosnik pa opazno odstopa od drugih.

In [ ]:
print(f'Število prenosnikov: {len(podatki)}')
print(f'Število atributov v izvirni tabeli: 9')
print(f'Število podvojenih povezav: {podatki.duplicated(subset=["url"]).sum()}')
print('\nMANJKAJOČE VREDNOSTI')
print(podatki[['ime', 'procesor', 'ram', 'ssd', 'zaslon', 'graficna',
               'operacijski_sistem', 'cena', 'url']].isna().sum())
print('\nOSNOVNE STATISTIKE')
podatki[['cena_eur', 'ram_gb', 'ssd_gb', 'zaslon_in']].describe().round(2)

## Porazdelitev cen

S histogramom sem prikazal, v katerih cenovnih razredih je največ prenosnikov. Največ naprav je zbranih okoli cen 999 €, 1.199 € in 1.499 €, kar kaže, da trgovina pogosto uporablja cene, ki se končajo na 99. Večina prenosnikov stane med približno 900 € in 1.500 €, nad 2.000 € pa so le posamezni zmogljivejši modeli.

In [ ]:
plt.figure(figsize=(12, 7))
plt.hist(podatki['cena_eur'], bins=12, color='steelblue', edgecolor='black', alpha=0.8)
plt.axvline(podatki['cena_eur'].mean(), color='red', linestyle='--',
            label=f"Povprečje: {podatki['cena_eur'].mean():.2f} €")
plt.axvline(podatki['cena_eur'].median(), color='green', linestyle='--',
            label=f"Mediana: {podatki['cena_eur'].median():.2f} €")
plt.title('Porazdelitev cen prenosnikov')
plt.xlabel('Cena (€)')
plt.ylabel('Število prenosnikov')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Najdražji in najcenejši prenosniki

Nato sem poiskal pet najdražjih in pet najcenejših naprav. Najdražji je Microsoft Surface Laptop Studio 2 s ceno 2.999 €, 32 GB RAM-a, 1 TB SSD-jem in grafično kartico RTX 4050. Najcenejša sta Dell Precision 7720 in Lenovo ThinkPad X1 Yoga G3, ki staneta 499 €.

Rezultat pokaže, da sama količina RAM-a ne določa cene. Najdražji model nima največ RAM-a, temveč na njegovo ceno vplivajo tudi novejši procesor, grafična kartica, zaslon in vrsta naprave.

In [ ]:
stolpci = ['ime', 'procesor', 'ram', 'ssd', 'graficna', 'cena_eur']

print('PET NAJDRAŽJIH PRENOSNIKOV')
display(podatki.nlargest(5, 'cena_eur')[stolpci].reset_index(drop=True))

print('PET NAJCENEJŠIH PRENOSNIKOV')
display(podatki.nsmallest(5, 'cena_eur')[stolpci].reset_index(drop=True))

## Vpliv količine RAM-a na ceno

Prenosnike sem združil glede na količino RAM-a in za vsako skupino izračunal povprečno ceno. Največ naprav ima 16 GB RAM-a, in sicer 38, sledi 26 naprav z 32 GB. Prenosniki z 8 GB RAM-a v povprečju stanejo 899 €, s 16 GB 1.066,11 €, z 32 GB 1.480,15 € in s 64 GB 1.739 €.

Povprečna cena se torej z večanjem količine RAM-a jasno povečuje. Razlika med povprečno ceno naprave s 16 GB in 32 GB RAM-a znaša približno 414 €. Vseeno iz tega ne moremo sklepati, da RAM sam povzroča razliko, saj imajo dražji prenosniki pogosto tudi boljše druge komponente.

In [ ]:
ram_statistika = podatki.groupby('ram_gb').agg(
    stevilo=('ime', 'size'),
    povprecna_cena=('cena_eur', 'mean'),
    mediana=('cena_eur', 'median')
).round(2)
display(ram_statistika)

plt.figure(figsize=(10, 6))
stolpci_ram = plt.bar(
    ram_statistika.index.astype(int).astype(str),
    ram_statistika['povprecna_cena'],
    color='cornflowerblue', edgecolor='black'
)
plt.bar_label(stolpci_ram, fmt='%.0f €', padding=3)
plt.title('Povprečna cena glede na količino RAM-a')
plt.xlabel('RAM (GB)')
plt.ylabel('Povprečna cena (€)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Vpliv velikosti SSD-ja na ceno

Pri SSD-jih sem vse vrednosti najprej pretvoril v gigabajte, pri čemer sem 1 TB zapisal kot 1000 GB. Najpogostejši je 512 GB SSD, ki ga ima 51 od 70 prenosnikov z znanim podatkom. Prenosniki z 256 GB SSD-jem stanejo povprečno 819 €, tisti s 512 GB 1.209,98 €, naprave z 1 TB pa 1.632,33 €.

Tudi tukaj opazimo rast povprečne cene z večanjem prostora za shranjevanje. Skupina s 500 GB vsebuje samo dve napravi, zato njeno povprečje ni dovolj zanesljivo za splošen sklep.

In [ ]:
ssd_statistika = podatki.dropna(subset=['ssd_gb']).groupby('ssd_gb').agg(
    stevilo=('ime', 'size'),
    povprecna_cena=('cena_eur', 'mean')
).round(2)
display(ssd_statistika)

plt.figure(figsize=(10, 6))
stolpci_ssd = plt.bar(
    ssd_statistika.index.astype(int).astype(str),
    ssd_statistika['povprecna_cena'],
    color='mediumseagreen', edgecolor='black'
)
plt.bar_label(stolpci_ssd, fmt='%.0f €', padding=3)
plt.title('Povprečna cena glede na velikost SSD-ja')
plt.xlabel('SSD (GB)')
plt.ylabel('Povprečna cena (€)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Namenska grafična kartica in cena

V podatkih ima 44 prenosnikov zapisano grafično kartico iz družine RTX ali Quadro, pri 29 pa namenska grafična kartica ni navedena. Prenosniki z namensko grafično kartico v povprečju stanejo 1.379,68 €, drugi pa 1.054,52 €. Razlika v povprečni ceni znaša približno 325 €.

To je pričakovan rezultat, saj so namenske grafične kartice namenjene zahtevnejšemu grafičnemu, inženirskemu ali profesionalnemu delu. Treba je upoštevati, da prazen podatek ne zagotavlja, da prenosnik nima nobene namenske grafične kartice; pomeni le, da je moj regularni izraz iz naslova izdelka ni izločil.

In [ ]:
gpu_statistika = podatki.groupby('namenska_graficna').agg(
    stevilo=('ime', 'size'),
    povprecna_cena=('cena_eur', 'mean'),
    mediana=('cena_eur', 'median')
).round(2)
gpu_statistika.index = ['Ni navedena', 'RTX ali Quadro']
display(gpu_statistika)

plt.figure(figsize=(9, 6))
stolpci_gpu = plt.bar(
    gpu_statistika.index, gpu_statistika['povprecna_cena'],
    color=['lightgray', 'darkorange'], edgecolor='black'
)
plt.bar_label(stolpci_gpu, fmt='%.0f €', padding=3)
plt.title('Povprečna cena glede na namensko grafično kartico')
plt.xlabel('Grafična kartica')
plt.ylabel('Povprečna cena (€)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Korelacija med tehničnimi lastnostmi in ceno

Za konec sem izračunal Pearsonove korelacijske koeficiente med ceno, RAM-om, SSD-jem in velikostjo zaslona. Korelacija meri linearno povezanost med dvema številskima spremenljivkama. Vrednost blizu 1 pomeni močno pozitivno povezanost, vrednost blizu 0 pa zelo šibko linearno povezanost.

Največjo povezanost s ceno ima količina RAM-a, kjer korelacijski koeficient znaša 0,560. Gre za zmerno pozitivno povezanost. Med SSD-jem in ceno je koeficient 0,467, kar pomeni šibko do zmerno pozitivno povezanost. Velikost zaslona ima s ceno koeficient samo 0,164, zato je njuna linearna povezanost zelo šibka.

Korelacija ne pomeni vzročnosti. Višja količina RAM-a ali večji SSD sta pogosto del sicer zmogljivejšega in novejšega prenosnika, zato na ceno hkrati vplivajo tudi procesor, grafična kartica, starost modela in kakovost izdelave.

In [ ]:
korelacije = podatki[['cena_eur', 'ram_gb', 'ssd_gb', 'zaslon_in']].corr().round(3)
display(korelacije)

pari = [
    ('ram_gb', 'RAM (GB)', 'royalblue'),
    ('ssd_gb', 'SSD (GB)', 'seagreen'),
    ('zaslon_in', 'Velikost zaslona (palci)', 'darkorange')
]

fig, osi = plt.subplots(1, 3, figsize=(17, 5))
for os, (stolpec, oznaka, barva) in zip(osi, pari):
    veljavni = podatki.dropna(subset=[stolpec, 'cena_eur'])
    os.scatter(veljavni[stolpec], veljavni['cena_eur'], color=barva, alpha=0.65)
    r = veljavni[[stolpec, 'cena_eur']].corr().iloc[0, 1]
    os.set_title(f'{oznaka} in cena (r = {r:.3f})')
    os.set_xlabel(oznaka)
    os.set_ylabel('Cena (€)')
    os.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Zaključek

Z analizo sem ugotovil, da spletna stran v izbrani kategoriji ponuja 73 različnih prenosnikov s procesorjem Intel Core i7. Tipičen prenosnik v podatkovni množici ima 16 GB RAM-a, 512 GB SSD in 15,6-palčni zaslon, njegova mediana cene pa znaša 1.199 €.

Najbolj očitna povezava je med količino RAM-a in ceno. Prenosniki z več RAM-a so v povprečju dražji, podobno velja tudi za večje SSD-je. Prenosniki z navedeno namensko grafično kartico RTX ali Quadro so v povprečju približno 325 € dražji od naprav, pri katerih taka kartica ni navedena. Velikost zaslona je bila s ceno povezana precej manj.

Pri interpretaciji je treba upoštevati, da podatki prikazujejo trenutno ponudbo samo ene spletne trgovine in samo modele s procesorjem Intel Core i7. Poleg tega na ceno vplivajo še generacija procesorja, starost naprave, stanje obnovljenega izdelka in kakovost izdelave, ki jih v tej analizi nisem podrobneje obravnaval. Analizo bi bilo mogoče nadgraditi z izločanjem generacije procesorja, primerjavo proizvajalcev ali spremljanjem spreminjanja cen skozi čas.